In [1]:
import pandas as pd
import datetime as dt

# 이전에 생성한 데이터를 불러옵니다.
df = pd.read_csv('data/customer_orders.csv')
df['OrderDate'] = pd.to_datetime(df['OrderDate'])

print("데이터가 성공적으로 로드되었습니다.")
df.info()

데이터가 성공적으로 로드되었습니다.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   OrderID          5000 non-null   object        
 1   CustomerID       5000 non-null   object        
 2   OrderDate        5000 non-null   datetime64[ns]
 3   ProductID        5000 non-null   object        
 4   ProductCategory  5000 non-null   object        
 5   Price            5000 non-null   int64         
dtypes: datetime64[ns](1), int64(1), object(4)
memory usage: 234.5+ KB


In [2]:
# RFM 계산을 위한 기준 날짜 설정
snapshot_date = dt.datetime(2025, 8, 1)

# RFM 값 계산
rfm_df = df.groupby('CustomerID').agg({
    'OrderDate': lambda date: (snapshot_date - date.max()).days, # 최근성 (Recency)
    'OrderID': 'nunique', # 구매 빈도 (Frequency)
    'Price': 'sum' # 총 구매 금액 (Monetary)
})

# 컬럼 이름 변경
rfm_df.rename(columns={'OrderDate': 'Recency',
                       'OrderID': 'Frequency',
                       'Price': 'Monetary'}, inplace=True)

print("RFM 값 계산 완료:")
rfm_df.head()

RFM 값 계산 완료:


,Recency,Frequency,Monetary
CustomerID,,,
C0001,86,4,180000
C0002,99,5,75000
C0003,199,4,121000
C0004,170,2,71000
C0005,116,4,155000


In [3]:
# 분위수(Quintiles)를 기준으로 점수 부여
r_labels = range(5, 0, -1) # Recency는 낮을수록 점수가 높아야 하므로 역순
f_labels = range(1, 6)
m_labels = range(1, 6)

rfm_df['R_Score'] = pd.qcut(rfm_df['Recency'], q=5, labels=r_labels, duplicates='drop').astype(int)
rfm_df['F_Score'] = pd.qcut(rfm_df['Frequency'], q=5, labels=f_labels, duplicates='drop').astype(int)
rfm_df['M_Score'] = pd.qcut(rfm_df['Monetary'], q=5, labels=m_labels, duplicates='drop').astype(int)

# RFM 점수 조합
rfm_df['RFM_Score'] = rfm_df['R_Score'].astype(str) + rfm_df['F_Score'].astype(str) + rfm_df['M_Score'].astype(str)

print("RFM 점수 부여 완료:")
rfm_df.head()

RFM 점수 부여 완료:


,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
CustomerID,,,,,,,
C0001,86,4,180000,3,2,3,323
C0002,99,5,75000,3,3,1,331
C0003,199,4,121000,2,2,2,222
C0004,170,2,71000,2,1,1,211
C0005,116,4,155000,3,2,3,323


In [4]:
# RFM 점수를 기반으로 세그먼트 정의
def segment_customer(df):
    if df['R_Score'] >= 4 and df['F_Score'] >= 4:
        return 'VIP'
    elif df['R_Score'] >= 3 and df['F_Score'] >= 3:
        return '충성 고객 (Loyal)'
    elif df['R_Score'] >= 3 or df['F_Score'] >= 3:
        return '잠재 고객 (Potential)'
    else:
        return '이탈 위험 고객 (At Risk)'

rfm_df['Segment'] = rfm_df.apply(segment_customer, axis=1)

print("세그먼트 분포 확인:")
print(rfm_df['Segment'].value_counts())
rfm_df.head()

세그먼트 분포 확인:
Segment
잠재 고객 (Potential)     340
이탈 위험 고객 (At Risk)    252
충성 고객 (Loyal)         206
VIP                   198
Name: count, dtype: int64


,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
CustomerID,,,,,,,,
C0001,86,4,180000,3,2,3,323,잠재 고객 (Potential)
C0002,99,5,75000,3,3,1,331,충성 고객 (Loyal)
C0003,199,4,121000,2,2,2,222,이탈 위험 고객 (At Risk)
C0004,170,2,71000,2,1,1,211,이탈 위험 고객 (At Risk)
C0005,116,4,155000,3,2,3,323,잠재 고객 (Potential)


In [6]:
# 결과 파일 저장
output_filename = 'data/customer_rfm.csv'
rfm_df.to_csv(output_filename, encoding='utf-8-sig')

print("-" * 30)
print(f"✅ RFM 분석 완료! 결과가 '{output_filename}' 파일로 저장되었습니다.")

------------------------------
✅ RFM 분석 완료! 결과가 'data/customer_rfm.csv' 파일로 저장되었습니다.
